# Sprint 4 · Qwen2.5-1.5B на CoAT-RU
Выберите T4 GPU и загрузите coat_ru.jsonl из публичного бенчмарка предыдущего спринта.


In [ ]:
import os, sys, subprocess
from pathlib import Path
os.environ['HF_HUB_DISABLE_IMPLICIT_TOKEN'] = '1'
ROOT = Path('/content/DW-1-sprint4')
if not ROOT.exists():
    subprocess.run(['git', 'clone', 'https://github.com/spotyyyi123-ux/DW-1.git', str(ROOT)], check=True)
os.chdir(ROOT)
subprocess.run(['git', 'checkout', '--detach', '4c5ae1dd4d84ca51ee03cabbe06c246d477caae9'], check=True)
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-r', 'scripts/external/requirements.txt'], check=True)
import torch
assert torch.cuda.is_available(), 'Выберите GPU в среде выполнения'
print(torch.cuda.get_device_name(0))


In [ ]:
from google.colab import files
uploaded = files.upload()
assert set(uploaded) == {'coat_ru.jsonl'}
Path('/content/coat_ru.jsonl').write_bytes(uploaded['coat_ru.jsonl'])


In [ ]:
cmd = [sys.executable, '-u', 'scripts/run_qwen_local.py',
       '--model', 'Qwen/Qwen2.5-1.5B', '--revision', '8faed761d45a263340a0528343f099c05c9a4323',
       '--device', 'cuda', '--max-tokens', '512', '--language', 'ru',
       '--benchmark', '/content/coat_ru.jsonl', '--out', '/content/qwen15_ru_scores.parquet']
with subprocess.Popen(cmd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True) as process:
    for line in process.stdout:
        print(line, end='', flush=True)
    if process.wait():
        raise RuntimeError(f'Код выхода: {process.returncode}')


In [ ]:
from google.colab import files
from zipfile import ZipFile, ZIP_DEFLATED
archive = Path('/content/dw1-sprint4-ru-scores.zip')
with ZipFile(archive, 'w', ZIP_DEFLATED) as z:
    for name in ('qwen15_ru_scores.parquet', 'qwen15_ru_scores.manifest.json'):
        z.write(archive.parent / name, arcname=name)
files.download(str(archive))
